# H200과 Ironwood의 티처 포싱 비교

이 노트북은 이 저장소에 설치한 `teacher_forcing` 패키지를 사용합니다. 소스와 CLI의 계측·비교 로직을 그대로 호출합니다. 기본 `demo`는 합성 점수만 사용하며 장비가 필요하지 않습니다.

| 모드 | 실행 장소 | 준비 및 결과 |
|---|---|---|
| `demo` | 분석 PC | 예시 결과표 확인 |
| `capture_gpu` | H200 | 기준 토큰 ID와 GPU 텐서 생성 |
| `capture_tpu` | Ironwood | 같은 체크포인트 및 H200의 case.json으로 TPU 텐서 생성 |
| `compare` | 분석 PC | case.json, gpu/, tpu/를 모아 summary.md 및 CSV 생성 |

저장소 루트에서 `python -m pip install -e '.[notebook]'`로 설치합니다. 수집용 커널은 해당 가속기의 PyTorch 런타임과 `capture` 추가 의존성이 필요합니다. [환경 준비와 결과 해석](../docs/runbook_ko.md)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
from teacher_forcing.report import build_report
from teacher_forcing.capture import capture_device

MODE = "demo"
ROOT = Path("runs/case01")  # 현재 작업 디렉터리 기준. 장비별로 같은 실험의 case.json 사용
MODEL_DIR = Path("/models/qwen-parity")
REPEATS = 3

## 무엇을 같은 위치에서 비교하나

프롬프트가 P, H200 기준 응답이 A B C라면, B를 채점할 때 양쪽 모두 P A를 입력한다. TPU가 X를 더 높게 평가해도 C를 채점할 때는 P A B를 쓴다. **강제로 입력한 토큰 ID의 일치**와 **그 직전 두 장비가 독립적으로 계산한 1위 후보의 일치**를 구분한다.

| 고정 문맥 | 읽을 값 | 다음 단계에 넣을 토큰 |
|---|---|---|
| P | 마지막 입력 위치의 레이어 출력과 A를 예측하는 전체 로짓 | A |
| P A | A 위치의 레이어 출력과 B를 예측하는 전체 로짓 | B |
| P A B | B 위치의 레이어 출력과 C를 예측하는 전체 로짓 | C |

**프리필**은 P A B를 causal mask로 한 번에 처리해 세 위치를 읽는다. 프롬프트 길이가 p이면 배열의 0번부터 셌을 때 첫 응답의 점수는 p−1 위치다. **KV 디코드**는 P 처리 결과로 A를 채점하고, 그 캐시를 유지하면서 A와 B를 차례로 넣는다. 따라서 KV 재사용을 검증하는 첫 점수는 **B의 점수**다. 응답이 1토큰으로 끝나면 KV 재사용 위치는 0개다.

각 Transformer 블록의 **잔차 합산이 끝난 출력**을 읽고, 최종 정규화 출력과 전체 어휘 로짓도 따로 저장한다. 모델의 블록 번호는 0부터, 보고서의 응답 위치는 1부터 시작한다.

## 최종 결과표 생성

열을 읽는 순서는 **자체 반복 편차 → 같은 문맥의 Top-1 → 처음 오차가 커지는 블록 → 서비스 영향**이다. 두 장비가 강제 입력한 토큰 자체는 항상 같다. 아래 일치율은 각 장비의 원본 로짓에서 구한 1위를 비교한다.

- **로짓 최대 절대차**는 같은 어휘 ID끼리 점수를 뺀 절댓값 중 가장 큰 값이다. **1·2위 차이**가 작을 때는 작은 오차도 선택을 바꿀 수 있다.
- **상대 L2 오차(%)**는 벡터 차이의 크기를 GPU 벡터 크기로 나눈 뒤 100을 곱한다. 분모가 0이면 공란으로 두고 절대차를 본다. 코사인 유사도는 두 벡터 방향이 같아도 크기 차이를 놓친다.
- **첫 불일치**는 같은 접두사에서 다음 토큰의 1위 ID가 처음 달라진 응답 위치다. 레이어는 오차가 처음 커진 후보 지점까지 알려 준다. 원인을 좁히려면 해당 블록에 같은 입력 벡터를 주고 다시 계산한다.
- `cached` 경로의 첫 응답 위치는 프롬프트 프리필이다. 보고서에서는 이 첫 위치를 따로 표시하고 KV 재사용 위치만 한 번 더 집계한다. 평균이나 백분율에는 유효 위치 수를 함께 표시한다.

In [ ]:
if MODE == "demo":
    display(Markdown(build_report()))
elif MODE in {"capture_gpu", "capture_tpu"}:
    capture_device(MODE.split("_")[1], ROOT, MODEL_DIR, repeats=REPEATS)
elif MODE == "compare":
    display(Markdown(build_report(ROOT, ROOT / "report")))
else:
    raise ValueError("demo, capture_gpu, capture_tpu, compare 중 하나를 선택하세요.")